# Метрики классификации и дисбаланс

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Строить матрицу ошибок и выбирать метрику классификации с учётом редкого класса.

## Какая ошибка важна

Долю верных ответов мы уже использовали в [пилоте](../../nearest-neighbors/selection-evaluation/), а библиотечный вызов — в [scikit-learn](../../../python/ml-tools/training-evaluation/). Теперь разберём, какие именно ошибки она скрывает. Подготовку и обучение заново не вводим: рассматриваем уже полученные прогнозы.

Классификатор присваивает объекту метку. Назовём интересующий класс положительным, с меткой 1. Для каждого объекта есть правильная метка и прогноз. Совпадения бывают двух видов: TP — правильно найденный положительный, TN — правильно отвергнутый отрицательный. Ошибки: FP — ложная тревога, FN — пропуск положительного. Эти четыре числа образуют матрицу ошибок.

Всего объектов N=TP+TN+FP+FN. При порядке классов [0,1] строки матрицы означают правильные ответы, столбцы — прогнозы: первая строка [TN,FP], вторая [FN,TP].

Accuracy=(TP+TN)/N отвечает, какая доля всех объектов угадана. При 99 отрицательных и одном положительном постоянный ответ 0 получает 99%, но не находит ни одного положительного. Поэтому метрику выбирают по смыслу задачи, а не по самому красивому числу.

## Разобранный пример: четыре клетки

Пусть TP=6, FP=2, FN=4, TN=88. **Precision**, точность положительных ответов, равна TP/(TP+FP)=6/8=0.75. **Recall**, полнота, равна TP/(TP+FN)=6/10=0.6. Первая спрашивает, сколько тревог верны, вторая — сколько реальных положительных найдены.

Эти же количества можно разложить по строкам правильных ответов и столбцам прогнозов:

| Ответ | Прогноз 0 | Прогноз 1 | Всего |
| --- | --- | --- | --- |
| 0 | TN=88 | FP=2 | 90 |
| 1 | FN=4 | TP=6 | 10 |
| Всего | 92 | 8 | 100 |

Например, знаменатель precision — восемь положительных прогнозов в столбце «Прогноз 1», а знаменатель recall — десять положительных ответов в строке с ответом 1.

F1=2TP/(2TP+FP+FN)=12/18=2/3 объединяет точность и полноту. Она не учитывает TN и не заменяет оценку цены ошибок. Если знаменатель выбранной метрики ноль, нужно явно задать соглашение и сообщить, почему метрика не информативна.

Для нескольких классов показатели считают по каждому классу, затем усредняют. Macro-среднее даёт классам одинаковый вес; взвешенное по числу объектов может скрыть провал редкой категории. В ноутбуке начнём с двух классов и вручную проверим числа до вызова библиотеки.

Например, у постоянного ответа 0 на наборе из 99 отрицательных и одного положительного полнота класса 0 равна 1, класса 1 — 0. Macro-recall равен $(1+0)/2=0.5$. Взвешенная полнота равна $(99\cdot1+1\cdot0)/100=0.99$: большая группа скрыла провал на малой.

В коде `confusion_matrix` строит таблицу, `precision_score` и `recall_score` считают соответствующие метрики. `labels=[0,1]` фиксирует порядок осей, `zero_division=0` задаёт соглашение при нулевом знаменателе; само отсутствие положительных прогнозов всё равно нужно отметить.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score
actual = [1, 1, 1, 0, 0, 0]
predicted = [1, 1, 0, 1, 0, 0]
# confusion_matrix: строки — истина, столбцы — прогноз; порядок классов задан явно.
table = confusion_matrix(actual, predicted, labels=[0, 1])
assert table.tolist() == [[2, 1], [1, 2]]
assert precision_score(actual, predicted, zero_division=0) == 2 / 3
assert recall_score(actual, predicted, zero_division=0) == 2 / 3
print(table)


## Типичные ошибки

Не переводите precision и accuracy одним неуточнённым словом «точность». Проверяйте порядок аргументов: сначала правильные метки, затем прогнозы.

## Задачи

### 1. Базовая

Что означает FN?

<details>
<summary>Подсказка</summary>

Положительный объект не найден.

</details>



### 2. Базовая

Какая метрика делит TP на число всех тревог?

<details>
<summary>Подсказка</summary>

Знаменатель TP+FP.

</details>



### 3. Прикладная

TP=3,FP=1,FN=2. Найдите precision и recall.

<details>
<summary>Подсказка</summary>

Знаменатели различаются.

</details>



### 4. Прикладная

Из 100 объектов один положительный. Всегда предсказываем 0. Найдите accuracy и recall.

<details>
<summary>Подсказка</summary>

Угаданы все отрицательные.

</details>



### 5. Олимпиадная
В выборке 100 объектов, из них 10 положительных. Accuracy равна 0.9. Постройте две матрицы ошибок с такой accuracy: одну с recall=0, другую с recall=1. Можно ли восстановить recall только по accuracy?

<details>
<summary>Подсказка</summary>

В обеих матрицах должно быть 90 верных ответов; распределите десять ошибок по-разному.

</details>



### 6. Олимпиадная
Ответы равны [1,1,0,0,0], все прогнозы — 0. Найдите accuracy, recall и precision. Библиотека сообщила precision=0 по своему соглашению. Означает ли это, что все положительные прогнозы ошибочны?

<details>
<summary>Подсказка</summary>

Сначала заполните TP, FP, FN и TN; отдельно проверьте знаменатели.

</details>



## Итоги и самопроверка

Восстановите четыре клетки из двух списков и объясните каждую метрику словами. Почему стоимость ложной тревоги может изменить выбор показателя?

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

Правильная метка 1, прогноз 0: пропуск.

### Задача 2

Precision — точность положительных прогнозов.

### Задача 3

Precision=3/4, recall=3/5. Первая относится к прогнозам, вторая к истинным положительным.

### Задача 4

Accuracy=99/100, recall=0/1=0. Высокая общая доля скрыла пропуск всего редкого класса.

### Задача 5

Первая: TP=0, FN=10, FP=0, TN=90. Accuracy=90/100=0.9, recall=0/10=0. Вторая: TP=10, FN=0, FP=10, TN=80. Accuracy=(10+80)/100=0.9, recall=10/10=1. Обе матрицы содержат 10 положительных и 90 отрицательных объектов. Одна и та же accuracy допускает противоположные значения полноты, поэтому восстановить её нельзя.

### Задача 6

TP=0, FP=0, FN=2, TN=3. Accuracy=3/5=0.6, recall=0/2=0. Precision=TP/(TP+FP)=0/0 не определена: положительных прогнозов вообще нет. Число 0, выбранное библиотекой для этого случая, позволяет вернуть числовой результат, но не превращает неопределённое отношение в долю ошибочных положительных прогнозов.